# **Обучение модели**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_sample_weight
from imblearn.over_sampling import SMOTE
from sklearn.base import clone
import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

Загрузим полученные предобработанные данные: 

In [40]:
df = pd.read_csv('../data/clean_data.csv')

TARGET_COL = 'Стадия'
y = df[TARGET_COL].astype(int)
X = df.drop(columns=[TARGET_COL, 'id'], errors='ignore')

print(f"Датасет: {X.shape[0]} пациентов, {X.shape[1]} признаков")

Датасет: 136 пациентов, 36 признаков


In [41]:
folds = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)

In [42]:
def train_cv(model, folds, folds_metrics, train_losses, val_losses):
    for fold, (train_idx, val_idx) in enumerate(folds.split(X, y), 1):
        fold_model = clone(model)
    
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
        fold_model.fit(
            X_tr, y_tr,
            eval_set=[(X_tr, y_tr), (X_val, y_val)],
            verbose=False
        )
    
        y_pred = fold_model.predict(X_val)
        folds_metrics.append({
            'accuracy': accuracy_score(y_val, y_pred),
            'f1_macro': f1_score(y_val, y_pred, average='macro')
        })
        
        evals = fold_model.evals_result()
        train_losses.append(evals['validation_0']['mlogloss'])
        val_losses.append(evals['validation_1']['mlogloss'])
        
        print(f"Fold {fold}: Acc={folds_metrics[-1]['accuracy']:.3f} | F1={folds_metrics[-1]['f1_macro']:.3f}")

    # Итоги по фолдам
    metrics_df = pd.DataFrame(folds_metrics)
    print(f"\nCV Accuracy: {metrics_df['accuracy'].mean():.3f}")
    print(f"CV Macro F1 : {metrics_df['f1_macro'].mean():.3f}")

### **1. Baseline**

В качестве моделе, будем использовать реализацию градиентного бустинга - `XGBoost`. Для валидации используется кросс-валидация с сохранением пропорции классов - `StratifiedKFold`.

In [43]:
base_model = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=7, 
    eval_metric='mlogloss',
    learning_rate=0.025, 
    max_depth=3, 
    n_estimators=500,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    verbosity=0,
    use_label_encoder=False
)

In [44]:
folds_metrics_base = []
train_losses_base, val_losses_base = [], []

In [45]:
train_cv(base_model, folds, folds_metrics_base, train_losses_base, val_losses_base)

Fold 1: Acc=0.853 | F1=0.827
Fold 2: Acc=0.824 | F1=0.732
Fold 3: Acc=0.588 | F1=0.485
Fold 4: Acc=0.647 | F1=0.619

CV Accuracy: 0.728
CV Macro F1 : 0.666


### **2. Подбор гиперпараметров**

Для подбора гиперпараметров `RandomSearch`, который перебирает случайные наборы параметров на сетке.

In [46]:
param_dist = {
    'max_depth': [2, 3, 4, 5, 6],
    'min_child_weight': [2, 3, 5],
    'learning_rate': [0.005, 0.01, 0.03, 0.05],
    'n_estimators': [300, 500, 1000],
    'subsample': [0.6, 0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8],
    'gamma': [0.1, 0.2, 0.3],
}

modelXGB = xgb.XGBClassifier(objective='multi:softprob', random_state=42)

random_search = RandomizedSearchCV(
    modelXGB, param_distributions=param_dist,
    n_iter=20,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='f1_macro',
    n_jobs=-1,
    random_state=42
)

random_search.fit(X, y)
print(random_search.best_params_)

{'subsample': 0.8, 'n_estimators': 500, 'min_child_weight': 2, 'max_depth': 5, 'learning_rate': 0.03, 'gamma': 0.1, 'colsample_bytree': 0.7}


In [47]:
best_model = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=7, 
    eval_metric='mlogloss',
    learning_rate=0.03, 
    max_depth=5, 
    n_estimators=500,
    subsample=0.8,
    colsample_bytree=0.7,
    min_child_weight=2,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    verbosity=0,
    use_label_encoder=False
)

In [48]:
folds_metrics_best = []
train_losses_best, val_losses_best = [], []

In [49]:
train_cv(best_model, folds, folds_metrics_best, train_losses_best, val_losses_best)

Fold 1: Acc=0.853 | F1=0.827
Fold 2: Acc=0.794 | F1=0.710
Fold 3: Acc=0.647 | F1=0.535
Fold 4: Acc=0.647 | F1=0.619

CV Accuracy: 0.735
CV Macro F1 : 0.673


### **3. Работа с несбалансированными классами**

In [50]:
base_accs, base_f1s = [], []

for train_idx, val_idx in folds.split(X, y):
    fold_model = clone(best_model)
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    fold_model.fit(X_tr, y_tr, eval_set=[(X_tr, y_tr), (X_val, y_val)], verbose=False)
    y_pred = fold_model.predict(X_val)
    
    base_accs.append(accuracy_score(y_val, y_pred))
    base_f1s.append(f1_score(y_val, y_pred, average='macro'))

#### **3.1. Class Weights**

In [51]:
cw_accs, cw_f1s = [], []

for train_idx, val_idx in folds.split(X, y):
    fold_model = clone(best_model)
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    w_tr = compute_sample_weight('balanced', y_tr)
    
    fold_model.fit(X_tr, y_tr, sample_weight=w_tr, 
                   eval_set=[(X_tr, y_tr), (X_val, y_val)], verbose=False)
    y_pred = fold_model.predict(X_val)
    
    cw_accs.append(accuracy_score(y_val, y_pred))
    cw_f1s.append(f1_score(y_val, y_pred, average='macro'))

#### **3.2. SMOTE**

In [52]:
smote_accs, smote_f1s = [], []
smote = SMOTE(k_neighbors=3, random_state=42)

for train_idx, val_idx in folds.split(X, y):
    fold_model = clone(best_model)
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    X_tr_res, y_tr_res = smote.fit_resample(X_tr, y_tr)
    
    fold_model.fit(X_tr_res, y_tr_res, 
                   eval_set=[(X_tr_res, y_tr_res), (X_val, y_val)], verbose=False)
    y_pred = fold_model.predict(X_val)
    
    smote_accs.append(accuracy_score(y_val, y_pred))
    smote_f1s.append(f1_score(y_val, y_pred, average='macro'))

In [53]:
results = pd.DataFrame({
    'Метод': ['Baseline', 'Class Weights', 'SMOTE'],
    'Accuracy': [
        f"{np.mean(base_accs):.3f} ± {np.std(base_accs):.3f}",
        f"{np.mean(cw_accs):.3f} ± {np.std(cw_accs):.3f}",
        f"{np.mean(smote_accs):.3f} ± {np.std(smote_accs):.3f}"
    ],
    'Macro F1': [
        f"{np.mean(base_f1s):.3f} ± {np.std(base_f1s):.3f}",
        f"{np.mean(cw_f1s):.3f} ± {np.std(cw_f1s):.3f}",
        f"{np.mean(smote_f1s):.3f} ± {np.std(smote_f1s):.3f}"
    ]
})

print("\n Итоговое сравнение методов балансировки:\n")
print(results.to_markdown(index=False))


 Итоговое сравнение методов балансировки:

| Метод         | Accuracy      | Macro F1      |
|:--------------|:--------------|:--------------|
| Baseline      | 0.735 ± 0.091 | 0.673 ± 0.108 |
| Class Weights | 0.721 ± 0.094 | 0.659 ± 0.101 |
| SMOTE         | 0.757 ± 0.113 | 0.700 ± 0.118 |


В ходе экспериментов были протестированы три подхода: базовая модель, взвешивание классов (Class Weights) и синтетическая передискретизация (SMOTE).

- Базовая модель показала стабильные результаты (Accuracy: 0.735, Macro F1: 0.673), но уступала в распознавании редких стадий.

- Class Weights не дали прироста качества (F1: 0.659), что может свидетельствовать о том, что простое взвешивание недостаточно для сложной границы решений в данном датасете.
    
- SMOTE показал наилучшие результаты (Accuracy: 0.757, Macro F1: 0.700). Прирост по Macro F1 составил +4%, что критично для медицинской диагностики, так как улучшилось распознавание редких стадий (1, 2, 7). Несмотря на увеличение дисперсии (std F1: 0.118), метод был выбран как финальный для обучения модели.